# 13_unseen_v2_calibration.ipynb

Task 5.8: dựng lại bộ họ chưa thấy với độ khó khớp theo ΔF1 của **ConvNeXt-Tiny** (mô hình thay thế), từ ImageNet-C (`imagecorruptions`) và ImageNet-C-bar (Mintun et al., R33). Chỉ suy luận, không train, **không mở tập test**.

Quy tắc đã đăng ký trước ở `registration` của `configs/unseen_families_v2.yaml` và không đổi trong notebook này (bản v2.1, xem `registration.amendments`):

1. Loại họ lỗi trong môi trường hoặc chậm hơn `max_seconds_per_image`.
2. Mục tiêu ΔF1 ba mức = các phân vị 0.2, 0.5, 0.8 của ΔF1 mô hình thay thế trên 42 điều kiện của benchmark 4.1 (validation).
3. Mỗi họ: chọn mức ứng viên gần mục tiêu nhất, sai lệch không quá dung sai. Họ không khớp bị bỏ, ghi lý do. Không nới dung sai, không chọn lại sau khi thấy kết quả.
4. Độ khó thực của Swin-Tiny trên mức đã chọn chỉ để báo cáo, không dùng để chọn.

**Bản v2.1.** Lượt đầu theo lưới v2.0 chỉ giữ 2 trên 20 họ, vì mức 1 của ImageNet-C đã quá mạnh so với mục tiêu ΔF1 nhẹ. v2.1 thêm năm mức nhẹ hơn cho ImageNet-C (trộn alpha ảnh gốc với mức 1: 0.1, 0.2, 0.35, 0.5, 0.7) và lấy seed ảnh suy giảm theo (họ, giá trị) để mức lúc chọn và lúc đo lại giống hệt nhau. Mục tiêu, dung sai, luật ba mức, mô hình thay thế đều giữ nguyên. Kết quả lượt đầu lưu ở `reports/unseen_v2/pass1_v2.0/`, notebook tự chuyển file cũ vào đó nếu thấy.

Kết quả ghi `reports/unseen_v2/calibration.json`. Notebook 14 (task 5.7) đọc file này, nên phải commit file lên nhánh trước khi chạy notebook 14.

## 1. Môi trường

Internet On, Accelerator GPU. Dataset input gồm `cthng123/durian-leaf-disease-dataset` và dataset chứa checkpoint của ConvNeXt-Tiny và Swin-Tiny (seed 42). Họ ngoài chạy trên CPU, nên nên bật cả 4 CPU của Kaggle.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Đổi nhánh tại đây, rồi chạy lại cell.
GIT_BRANCH = "unseen-families-v2-5.5-5.7-5.8"
REPO_URL = "https://github.com/meo225/durian-leaf-disease-robustness.git"
REPO_DIR = Path("/kaggle/working") / "durian-leaf-disease-robustness"

if not (Path("/kaggle/working").is_dir() and Path("/kaggle/input").is_dir()):
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle. Xem README, mục Chạy trên Kaggle.")

def git(args, cwd=None):
    subprocess.run(
        ["git", *args],
        cwd=cwd,
        check=True,
        env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
    )


if not (REPO_DIR / ".git").is_dir():
    git(["clone", "--branch", GIT_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)])
else:
    git(["fetch", "origin", GIT_BRANCH], cwd=REPO_DIR)
    git(["checkout", GIT_BRANCH], cwd=REPO_DIR)
    git(["reset", "--hard", f"origin/{GIT_BRANCH}"], cwd=REPO_DIR)

matches = []
seen = set()
for path in Path("/kaggle/input").rglob("DLD_FinalDataset_224_spit"):
    resolved = str(path.resolve())
    if path.is_dir() and resolved not in seen:
        seen.add(resolved)
        matches.append(path)

if not matches:
    found = []
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for path in sorted(input_root.rglob("*")):
            if path.is_dir() and len(path.relative_to(input_root).parts) <= 3:
                found.append(str(path.relative_to(input_root)))
    listing = "\n".join(found[:40]) if found else "(trống)"
    raise FileNotFoundError(
        "Chưa thấy DLD_FinalDataset_224_spit trong /kaggle/input.\n"
        "Add Input: cthng123/durian-leaf-disease-dataset, rồi Save & Run All.\n"
        f"Thư mục hiện có:\n{listing}"
    )

image_root = matches[0].parent
link = REPO_DIR / "datasets" / "durian-ldd"
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink() or link.is_file():
    link.unlink()
elif link.exists():
    link.rmdir()
link.symlink_to(image_root, target_is_directory=True)

if not (link / "DLD_FinalDataset_224_spit").is_dir():
    raise FileNotFoundError(f"Không thấy ảnh tại {link}")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True
).strip()
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=REPO_DIR, text=True
).strip()
print(f"Nhánh: {branch}")
print(f"Commit: {commit}")
print(f"Ảnh: {link / 'DLD_FinalDataset_224_spit'}")

In [ ]:
import subprocess
import sys

# imagecorruptions 1.1.2 import pkg_resources (setuptools dưới 81). Một số họ lỗi với numpy mới, probe_families ở dưới sẽ loại.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagecorruptions==1.1.2", "setuptools<81"], check=True)
import numpy, scipy, skimage, torch, torchvision, timm
print("numpy", numpy.__version__, "| scipy", scipy.__version__, "| scikit-image", skimage.__version__)
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| timm", timm.__version__)

## 2. Checkpoint

Mô hình thay thế dùng để khớp mức, mô hình tham chiếu chỉ để báo độ khó thực.

In [ ]:
import yaml

from src.corruption import load_seed
from src.corruption.benchmark import load_config
from src.evaluation import unseen_families_v2 as U
from src.evaluation.reference import collect_checkpoint

CONFIG = U.CONFIG_PATH
if load_config(CONFIG)["locked"]:
    raise RuntimeError("configs/unseen_families_v2.yaml đã khóa. Không chạy lại hiệu chỉnh: đổi quy tắc nghĩa là đăng ký bộ mới (version 3), không sửa bộ này.")
registration = U.load_registration(CONFIG)
config_seed = load_seed(CONFIG)
pipeline_cfg = yaml.safe_load((REPO_DIR / "configs/pipeline.yaml").read_text(encoding="utf-8"))
CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
checkpoints = {
    role: collect_checkpoint("/kaggle/input", registration[role]["checkpoint"], CHECKPOINT_DIR)
    for role in ("surrogate", "reference")
}
for role, path in checkpoints.items():
    print(f"{role}: {registration[role]['model']}  {path.name}  {path.stat().st_size / 1024**2:.1f} MB")

## 3. Chạy thử các họ

Mỗi họ chạy thử trên một ảnh tổng hợp 224×224 và đo thời gian. Họ lỗi hoặc chậm bị loại theo quy tắc đã đăng ký. Không vá họ lỗi.

In [ ]:
import json

import pandas as pd

from src.corruption.external import probe_families

OUT_DIR = REPO_DIR / "reports/unseen_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)
probe = probe_families(size=int(pipeline_cfg["image_size"]))
eligible, excluded = U.eligible_candidates(probe, registration["candidates"]["max_seconds_per_image"], registration["candidates"]["sources"])
table = pd.DataFrame(probe).T
table["trạng thái"] = ["dùng" if name in eligible else "loại" for name in table.index]
display(table.sort_values("seconds_per_image"))
print(f"Giữ {len(eligible)} họ, loại {len(excluded)}.")
for name, reason in excluded.items():
    print(f"  loại {name}: {reason}")

## 4. Mục tiêu ΔF1 từ benchmark đã thấy

ΔF1 = macro F1 sạch trừ macro F1 suy giảm, đo trên validation bằng mô hình thay thế, trên 42 điều kiện của `configs/corruption.yaml`.

In [ ]:
import torch

from src.evaluation.difficulty import target_levels
from src.evaluation.reference import load_trained_model
from src.evaluation.robustness import evaluate_conditions
from src.corruption import load_conditions
from src.training.loop import resolve_device

device = resolve_device()
NUM_WORKERS = os.cpu_count() or 1
num_classes = len(pipeline_cfg["classes"])
models = {
    role: load_trained_model(yaml.safe_load((REPO_DIR / registration[role]["train_config"]).read_text(encoding="utf-8")), checkpoints[role], num_classes, device)
    for role in ("surrogate", "reference")
}
diff_cfg = registration["difficulty"]
seen_conditions = load_conditions(registration["seen_benchmark"], diff_cfg["split"])
seen_result = evaluate_conditions(models["surrogate"], pipeline_cfg, diff_cfg["split"], seen_conditions, device, load_seed(registration["seen_benchmark"]))
seen_drop = {name: entry["degradation"] for name, entry in seen_result["conditions"].items()}
targets = target_levels(seen_drop, diff_cfg["quantiles"])
print(f"Macro F1 sạch của mô hình thay thế: {seen_result['clean']['macro_f1']:.4f}")
print("ΔF1 trên 42 điều kiện đã thấy: nhỏ nhất {:.3f}, trung vị {:.3f}, lớn nhất {:.3f}".format(min(seen_drop.values()), sorted(seen_drop.values())[len(seen_drop) // 2], max(seen_drop.values())))
print("Mục tiêu ΔF1 ba mức:", [round(t, 3) for t in targets], "| dung sai", diff_cfg["tolerance"])

## 5. Đo mọi mức ứng viên

Mỗi (họ, mức) là một lượt suy luận trên 444 ảnh validation. Kết quả ghi dần vào `candidates_surrogate.json` theo từng điều kiện, chạy lại cell thì tiếp tục từ chỗ dừng. File của lượt đầu (không có `seed_rule`) được chuyển sang `pass1_v2.0/`, vì seed khác nên không dùng lại số cũ. Ước tính khoảng 1 giờ, phần lớn ở bốn họ ImageNet-C-bar.

In [ ]:
import shutil
import time

from src.corruption.external import SEED_RULE

cache_path = OUT_DIR / "candidates_surrogate.json"
cache = {"seed_rule": SEED_RULE, "conditions": {}}
if cache_path.exists():
    previous = json.loads(cache_path.read_text(encoding="utf-8"))
    if previous.get("seed_rule") == SEED_RULE:
        cache = previous
    else:
        archive = OUT_DIR / "pass1_v2.0"
        archive.mkdir(exist_ok=True)
        for name in ("candidates_surrogate.json", "calibration.json", "realised_difficulty.csv"):
            if (OUT_DIR / name).exists() and not (archive / name).exists():
                shutil.move(str(OUT_DIR / name), str(archive / name))
        print("File lượt đầu (v2.0, seed theo tên điều kiện) đã chuyển sang", archive.relative_to(REPO_DIR))
candidates = U.candidate_conditions(eligible, registration)
config_seed = load_seed(CONFIG)
for family in eligible:
    members = {name: spec for name, spec in candidates.items() if spec["family"] == family}
    pending = {name: spec for name, spec in members.items() if name not in cache["conditions"]}
    if not pending:
        continue
    start = time.time()
    result = evaluate_conditions(models["surrogate"], pipeline_cfg, diff_cfg["split"], pending, device, config_seed, num_workers=NUM_WORKERS)
    for name, entry in result["conditions"].items():
        cache["conditions"][name] = {"family": family, "value": pending[name]["value"], "macro_f1": entry["macro_f1"], "delta_f1": entry["degradation"]}
    cache_path.write_text(json.dumps(cache, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"{family:32s} {len(pending):2d} mức {time.time() - start:6.0f} s")
delta_table = {family: {} for family in eligible}
for entry in cache["conditions"].values():
    if entry["family"] in delta_table:
        delta_table[entry["family"]][entry["value"]] = entry["delta_f1"]
display(pd.DataFrame(delta_table).T.sort_index(axis=1).round(3))

## 6. Khớp mức

Theo `difficulty.match_families`. Họ không có mức nào trong dung sai của một mục tiêu bị bỏ.

In [ ]:
from src.evaluation.difficulty import match_families

kept, dropped = match_families(delta_table, targets, diff_cfg["tolerance"])
display(pd.DataFrame(
    [{"họ": f, "mức": m["values"], "ΔF1": [round(d, 3) for d in m["delta_f1"]], "lệch": [round(e, 3) for e in m["errors"]]} for f, m in kept.items()]
))
print(f"Giữ {len(kept)} họ, bỏ {len(dropped)} họ do không khớp độ khó.")
for family, reason in dropped.items():
    print(f"  bỏ {family}: {reason}")

## 7. Độ khó thực trên mức đã chọn (chỉ báo cáo)

Chạy cả hai mô hình trên đúng điều kiện `<họ>_s<mức>` sẽ nằm trong config khóa. Mô hình thay thế phải gần mục tiêu. Swin-Tiny cho biết bộ này khó thế nào với mô hình tham chiếu, và **không** được dùng để chỉnh mức.

In [ ]:
conditions = U.matched_conditions(kept)
realised = {}
for role in ("surrogate", "reference"):
    result = evaluate_conditions(models[role], pipeline_cfg, diff_cfg["split"], conditions, device, config_seed, num_workers=NUM_WORKERS)
    realised[role] = {"clean_macro_f1": result["clean"]["macro_f1"], "conditions": {n: {"macro_f1": e["macro_f1"], "delta_f1": e["degradation"]} for n, e in result["conditions"].items()}}
rows = []
for family, match in kept.items():
    for level, target in enumerate(targets, start=1):
        name = f"{family}_s{level}"
        rows.append({
            "họ": family, "mức": level, "giá trị": match["values"][level - 1], "mục tiêu": round(target, 3),
            "ΔF1 thay thế (lúc chọn)": round(match["delta_f1"][level - 1], 3),
            "ΔF1 thay thế (thực)": round(realised["surrogate"]["conditions"][name]["delta_f1"], 3),
            "ΔF1 Swin-Tiny (thực)": round(realised["reference"]["conditions"][name]["delta_f1"], 3),
        })
realised_table = pd.DataFrame(rows)
display(realised_table)
for role in ("surrogate", "reference"):
    values = realised_table["ΔF1 thay thế (thực)" if role == "surrogate" else "ΔF1 Swin-Tiny (thực)"]
    print(f"{role}: ΔF1 trung bình {values.mean():.3f}, nhỏ nhất {values.min():.3f}, lớn nhất {values.max():.3f}")
print("Benchmark đã thấy, mô hình thay thế: ΔF1 trung bình {:.3f}".format(sum(seen_drop.values()) / len(seen_drop)))

## 8. Ghi kết quả

Ghi mọi thứ cần để truy vết: kết quả chạy thử, họ giữ và loại, mục tiêu, bảng ΔF1 ứng viên, độ khó thực.

In [ ]:
record = {
    "split": diff_cfg["split"],
    "config": CONFIG,
    "targets": targets,
    "tolerance": diff_cfg["tolerance"],
    "quantiles": diff_cfg["quantiles"],
    "seen_surrogate_delta_f1": seen_drop,
    "surrogate_clean_macro_f1": seen_result["clean"]["macro_f1"],
    "probe": probe,
    "eligible": eligible,
    "excluded_by_probe": excluded,
    "candidate_delta_f1": {family: {str(v): d for v, d in values.items()} for family, values in delta_table.items()},
    "kept": kept,
    "dropped_by_difficulty": dropped,
    "realised": realised,
    "seed_rule": SEED_RULE,
    "amendments": registration.get("amendments"),
}
(OUT_DIR / "calibration.json").write_text(json.dumps(record, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
realised_table.to_csv(OUT_DIR / "realised_difficulty.csv", index=False)
print("Đã ghi reports/unseen_v2/calibration.json")

## 9. Ảnh mẫu

Một ảnh validation thuộc mỗi lớp trong ba lớp, mỗi họ một dòng, cột đầu là ảnh sạch rồi ba mức. Cần nhìn bằng mắt: họ nào trông như không phải ảnh lá thật hoặc làm mất hẳn vết bệnh thì ghi nhận xét vào báo cáo, nhưng **không** đổi mức.

In [ ]:
from IPython.display import Image, display

from src.corruption import example_rows
from src.data.splits import get_split
from src.evaluation import plot_corruption_examples

val_classes = list(get_split("val")["class_name"])
for class_name in ("HEALTHY_LEAF", "LEAF_BLIGHT", "ALGAL_LEAF_SPOT"):
    rows = example_rows(pipeline_cfg, "val", conditions, config_seed, index=val_classes.index(class_name))
    figure_path = OUT_DIR / f"severity_examples_{class_name.lower()}.png"
    plot_corruption_examples(rows, figure_path)
    print(class_name)
    display(Image(filename=figure_path))

## 10. Nén kết quả

Tải `unseen_v2_calibration_outputs.zip` về, giải nén vào repo (giữ nguyên đường dẫn), commit `reports/unseen_v2/` lên nhánh rồi mới chạy notebook 14.

In [ ]:
import zipfile

for path in sorted(OUT_DIR.iterdir()):
    print(f"  {path.name} ({path.stat().st_size / 1024:.1f} KB)")
archive = Path("/kaggle/working/unseen_v2_calibration_outputs.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUT_DIR.rglob("*")):
        if path.is_file():
            bundle.write(path, path.relative_to(REPO_DIR).as_posix())
print(f"Đã nén {archive} ({archive.stat().st_size / 1024**2:.1f} MB)")